# Chạy toàn bộ Research Assist trên Kaggle

Notebook này cài và chạy chính backend + giao diện Research Assist.
Tải notebook và gói mã nguồn từ cùng một bản. Trên máy dự án, tạo ZIP bằng:

```powershell
.\.venv\Scripts\python.exe scripts/package_source.py
```

Đưa `dist/ResearchDesk_Source.zip` vào một Kaggle Dataset **riêng tư** rồi chọn **Add Input**.
Notebook lấy `notebooks/boot.sh` từ ZIP để khởi động backend.
Bật Internet để cài thư viện. Tạo và bật Kaggle Secret `RA_ACCESS_TOKEN` gồm 32–256 ký tự
URL-safe ngẫu nhiên (chữ, số, `_`, `-`). Dùng secret này để đăng nhập web.
Nếu chọn Gemini, cần thêm `RA_GEMINI_API_KEY`.
Literature Scout có thể tìm OpenAlex ngay. Để dùng hạn mức theo tài khoản, thêm
`RA_OPENALEX_API_KEY` vào Kaggle Secrets và bật `USE_OPENALEX_KEY`.
GPU chỉ cần thiết khi chọn embedding/reranker CUDA. Mặc định Demo giúp thử luồng
tải tài liệu, đặt câu hỏi và mở trích dẫn mà không gọi API mô hình.

Chạy các cell theo thứ tự trong phiên tương tác. Nếu được phép mở Cloudflare Quick Tunnel,
đặt cả `START_WORKSPACE` và `OPEN_PUBLIC_TUNNEL` thành `True`, rồi mở URL ở cell khởi động.
Dừng bằng cell cuối. URL và tiến trình sống cùng phiên Kaggle.
SQLite, tài liệu và cấu hình riêng nằm trong `/kaggle/temp/research-assist-data` trong phiên này.

In [ ]:
from pathlib import Path

SOURCE_ZIP = ""  # Trống: tự tìm đúng một ResearchDesk_Source.zip trong Add Input.
DATA_DIR = Path("/kaggle/temp/research-assist-data")  # Riêng tư, không đưa vào Kaggle Output.
PORT = 8000
START_WORKSPACE = False
OPEN_PUBLIC_TUNNEL = False
USE_GEMINI = False
GEMINI_MODEL = ""  # Trống: dùng model mặc định trong ứng dụng.
USE_OPENALEX_KEY = False
USE_GPU_RETRIEVAL = False
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"  # Hub ID hoặc thư mục model tương thích.
RERANKER_MODEL = ""  # Tùy chọn CrossEncoder tương thích, dùng cùng GPU.

In [ ]:
"""Start and stop an optional Cloudflare Quick Tunnel in an interactive notebook."""

import hashlib
import json
import os
import platform
import queue
import re
import shutil
import subprocess
import tempfile
import threading
import time
import urllib.request
from pathlib import Path


def find_cloudflared(working_dir: Path | str) -> str:
    installed = shutil.which("cloudflared")
    if installed:
        return installed
    machine = platform.machine().lower()
    if platform.system() != "Linux" or machine not in {"x86_64", "amd64", "aarch64", "arm64"}:
        raise RuntimeError("Install cloudflared for your operating system and put it on PATH.")
    arch = "amd64" if machine in {"x86_64", "amd64"} else "arm64"
    name = f"cloudflared-linux-{arch}"
    directory = Path(working_dir)
    directory.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(
        "https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
        headers={"User-Agent": "Research-Assist-Notebook"},
    )
    with urllib.request.urlopen(request, timeout=15) as response:
        release = json.load(response)
    asset = next((item for item in release["assets"] if item["name"] == name), None)
    if not asset or not re.fullmatch(r"sha256:[a-fA-F0-9]{64}", asset.get("digest") or ""):
        raise RuntimeError("No published SHA256 digest for cloudflared. Install it manually.")
    digest = asset["digest"].split(":", 1)[1].lower()
    target = directory / name
    if target.is_file():
        with target.open("rb") as existing:
            if hashlib.file_digest(existing, "sha256").hexdigest() == digest:
                target.chmod(0o700)
                return str(target.resolve())
    deadline = time.monotonic() + 90
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=directory, delete=False) as output:
            temporary = Path(output.name)
            checksum = hashlib.sha256()
            with urllib.request.urlopen(asset["browser_download_url"], timeout=15) as response:
                while block := response.read(1024 * 1024):
                    if time.monotonic() >= deadline:
                        raise TimeoutError("Downloading cloudflared took too long.")
                    checksum.update(block)
                    output.write(block)
        if checksum.hexdigest() != digest:
            raise RuntimeError("cloudflared does not match the official release checksum.")
        temporary.chmod(0o700)
        os.replace(temporary, target)
        return str(target.resolve())
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def stop_process(process: subprocess.Popen) -> None:
    """Reap the process even if it ignores a graceful stop."""
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)


def start_quick_tunnel(
    port: int, working_dir: Path | str, *, timeout: float = 60
) -> tuple[subprocess.Popen, str]:
    """Return the owned process and assigned HTTPS URL, or tear it down on error."""
    if not 1 <= port <= 65535 or timeout <= 0:
        raise ValueError("Invalid tunnel port or timeout.")
    executable = find_cloudflared(working_dir)
    process = subprocess.Popen(
        [executable, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    results: queue.Queue[str | None] = queue.Queue()

    def read_output() -> None:
        stream = process.stdout
        if stream is None:
            results.put(None)
            return
        found = False
        with stream:
            for line in stream:
                match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com\b", line)
                if not found and match:
                    found = True
                    results.put(match.group())
        results.put(None)

    try:
        threading.Thread(target=read_output, daemon=True).start()
        try:
            url = results.get(timeout=timeout)
        except queue.Empty:
            raise TimeoutError("No Cloudflare URL received before the timeout.") from None
        if not url or process.poll() is not None:
            raise RuntimeError("cloudflared exited before the tunnel was ready.")
        return process, url
    except BaseException:
        stop_process(process)
        raise

In [ ]:
"""Shared runtime of the Research Assist workspace notebook (stdlib only)."""

import http.client
import json
import os
import re
import socket
import subprocess
import time
import zipfile
from pathlib import Path, PurePosixPath
from urllib.parse import urlsplit

WORKSPACE_BOOT = Path("notebooks/boot.sh")


def unpack_workspace(archive_path: Path, destination: Path) -> Path:
    """Unpack a source-only archive into a new directory."""
    if destination.exists():
        raise FileExistsError("Choose a new directory for the source archive.")
    with zipfile.ZipFile(archive_path) as archive:
        files = {}
        for member in archive.infolist():
            if member.is_dir():
                continue
            parts = PurePosixPath(member.filename).parts
            if parts and parts[0] == "research-assist":
                parts = parts[1:]
            if (
                not parts
                or member.filename.startswith("/")
                or any(part in {"..", "."} or "\\" in part or ":" in part for part in parts)
            ):
                raise ValueError("Invalid path in the workspace archive.")
            if parts not in {("pyproject.toml",), ("LICENSE",), WORKSPACE_BOOT.parts} and (
                parts[:2] != ("src", "research_assist")
                or any(part.startswith(".") or part == "__pycache__" for part in parts)
            ):
                continue
            path = destination.joinpath(*parts)
            if path in files:
                raise ValueError("Duplicate file in workspace archive.")
            files[path] = member
        required = {destination / "pyproject.toml", destination / "src/research_assist/cli.py"}
        if not required.issubset(files):
            raise ValueError("Archive does not contain Research Assist source.")
        if sum(member.file_size for member in files.values()) > 100 * 1024**2:
            raise ValueError("Workspace source archive exceeds 100 MiB when unpacked.")
        for path, member in files.items():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(member))
    return destination


def workspace_command(source: Path, python: str, port: int) -> list[str]:
    launcher = source / WORKSPACE_BOOT
    if launcher.is_file():
        return ["bash", str(launcher), python, str(port)]
    return [
        python,
        "-m",
        "research_assist.cli",
        "serve",
        "--host",
        "127.0.0.1",
        "--port",
        str(port),
    ]


def kaggle_secret(name: str, *, min_length: int = 1) -> str:
    from kaggle_secrets import UserSecretsClient

    try:
        value = UserSecretsClient().get_secret(name).strip()
    except Exception:
        raise ValueError(f"Add and enable the Kaggle Secret {name}.") from None
    if len(value) < min_length or any(c in value for c in "\r\n"):
        raise ValueError(f"{name} must be a single line of at least {min_length} characters.")
    return value


def workspace_environment(
    data_dir: Path,
    access_token: str,
    public_url: str | None,
    *,
    gemini_key: str = "",
    gemini_model: str = "",
    openalex_key: str = "",
    use_gpu: bool = False,
    embedding_model: str = "intfloat/multilingual-e5-small",
    reranker_model: str = "",
) -> dict[str, str]:
    if not re.fullmatch(r"[A-Za-z0-9_-]{32,256}", access_token):
        raise ValueError("RA_ACCESS_TOKEN must be a URL-safe token of 32–256 characters.")
    hosts = ["localhost", "127.0.0.1"]
    if public_url:
        if not re.fullmatch(r"https://[a-z0-9-]+\.trycloudflare\.com", public_url):
            raise ValueError("Expected a Cloudflare Quick Tunnel HTTPS URL.")
        hostname = urlsplit(public_url).hostname
        if hostname is not None:
            hosts.append(hostname)

    env = {name: value for name, value in os.environ.items() if not name.startswith("RA_")}
    env.pop("PYTHONPATH", None)
    env.update(
        RA_ACCESS_TOKEN=access_token,
        RA_ALLOWED_HOSTS=json.dumps(hosts),
        RA_DATA_DIR=str(data_dir),
        RA_LLM_PROVIDER="gemini" if gemini_key else "demo",
        RA_LLM_MODEL=gemini_model if gemini_key else "",
        RA_GEMINI_API_KEY=gemini_key,
        RA_OPENALEX_API_KEY=openalex_key,
        RA_EMBEDDING_PROVIDER="sentence-transformers" if use_gpu else "none",
        RA_EMBEDDING_MODEL=embedding_model,
        RA_LOCAL_DEVICE="cuda" if use_gpu else "cpu",
        RA_RERANKER_PROVIDER="local" if use_gpu and reranker_model else "none",
        RA_RERANKER_MODEL=reranker_model if use_gpu else "",
        TOKENIZERS_PARALLELISM="false",
    )
    return env


def workspace_check(url: str, path: str, token: str = "") -> int:
    """Probe an explicitly selected origin without following redirects."""
    target = urlsplit(url)
    connection_cls = (
        http.client.HTTPSConnection if target.scheme == "https" else http.client.HTTPConnection
    )
    with_connection = connection_cls(target.hostname, target.port, timeout=5)
    try:
        headers = {"Authorization": "Bearer " + token} if token else {}
        with_connection.request("GET", path, headers=headers)
        response = with_connection.getresponse()
        return response.status
    finally:
        with_connection.close()


def wait_for_workspace(
    process: subprocess.Popen,
    base_url: str,
    token: str,
    *,
    timeout: float = 90,
) -> None:
    deadline = time.monotonic() + timeout
    while process.poll() is None and time.monotonic() < deadline:
        try:
            if workspace_check(base_url, "/api/health") == 200:
                if workspace_check(base_url, "/api/documents") != 401:
                    raise RuntimeError("Workspace refused to start without authentication.")
                if workspace_check(base_url, "/api/documents", token) == 200:
                    return
        except (OSError, http.client.HTTPException):
            pass
        time.sleep(0.5)
    raise RuntimeError("Workspace did not pass its startup and authentication checks.")


def check_loopback_port(port: int) -> None:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        try:
            probe.bind(("127.0.0.1", port))
        except OSError:
            raise RuntimeError(
                f"Port {port} is in use. Stop the old server or pick another port."
            ) from None

## Nạp và cài mã nguồn

In [ ]:
if START_WORKSPACE:
    import sys
    from uuid import uuid4

    if sys.version_info < (3, 11):
        raise RuntimeError("Research Assist requires Python 3.11 or newer.")
    access_token = kaggle_secret("RA_ACCESS_TOKEN", min_length=32)
    gemini_key = kaggle_secret("RA_GEMINI_API_KEY") if USE_GEMINI else ""
    openalex_key = kaggle_secret("RA_OPENALEX_API_KEY") if USE_OPENALEX_KEY else ""
    if USE_GPU_RETRIEVAL:
        import torch

        if not torch.cuda.is_available():
            raise RuntimeError("Enable a GPU accelerator before selecting CUDA retrieval.")
    if SOURCE_ZIP:
        source_zip = Path(SOURCE_ZIP)
    else:
        matches = sorted(Path("/kaggle/input").rglob("ResearchDesk_Source.zip"))
        if len(matches) != 1:
            raise RuntimeError("Add one ResearchDesk_Source.zip as Input or set SOURCE_ZIP.")
        (source_zip,) = matches
    work_root = Path("/kaggle/temp/research-assist-runtime")
    work_root.mkdir(parents=True, exist_ok=True)
    source = unpack_workspace(source_zip, work_root / ("source-" + uuid4().hex))
    env_dir = work_root / "venv"
    # Isolate app packages while reusing the Kaggle image's CUDA PyTorch.
    subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(env_dir)],
        check=True,
        timeout=180,
    )
    workspace_python = str(env_dir / "bin/python")
    package = str(source) + ("[local-models]" if USE_GPU_RETRIEVAL else "")
    subprocess.run(
        [workspace_python, "-m", "pip", "install", package],
        cwd=work_root,
        check=True,
        timeout=900,
    )
else:
    print("Đặt START_WORKSPACE=True trong cell cấu hình khi sẵn sàng.")

## Khởi động và lấy URL

In [ ]:
if START_WORKSPACE:
    if "workspace_python" not in globals():
        raise RuntimeError("Run the source installation cell first.")
    if "workspace_process" in globals() and workspace_process.poll() is None:
        raise RuntimeError("Stop the current workspace using the last cell before restarting.")
    if "workspace_tunnel" in globals() and workspace_tunnel and workspace_tunnel.poll() is None:
        raise RuntimeError("Stop the current tunnel using the last cell before restarting.")
    check_loopback_port(PORT)
    local_url = f"http://127.0.0.1:{PORT}"
    public_url = None
    workspace_tunnel = workspace_process = None
    try:
        if OPEN_PUBLIC_TUNNEL:
            workspace_tunnel, public_url = start_quick_tunnel(PORT, work_root)
        env = workspace_environment(
            DATA_DIR, access_token, public_url, gemini_key=gemini_key,
            gemini_model=GEMINI_MODEL, openalex_key=openalex_key, use_gpu=USE_GPU_RETRIEVAL,
            embedding_model=EMBEDDING_MODEL, reranker_model=RERANKER_MODEL,
        )
        log_path = work_root / "workspace.log"
        with log_path.open("w", encoding="utf-8") as log:
            workspace_process = subprocess.Popen(
                workspace_command(source, workspace_python, PORT),
                cwd=work_root,
                env=env,
                stdout=log,
                stderr=subprocess.STDOUT,
            )
        print("Đang kiểm tra ứng dụng và xác thực. Log:", log_path)
        wait_for_workspace(workspace_process, local_url, access_token)
        if public_url:
            wait_for_workspace(workspace_process, public_url, access_token, timeout=90)
            print("Mở Research Assist:", public_url)
            print("Đăng nhập bằng giá trị RA_ACCESS_TOKEN từ Kaggle Secrets.")
        else:
            print("Workspace hoạt động trong phiên Kaggle:", local_url)
            print("Để truy cập từ trình duyệt bên ngoài, dừng rồi bật OPEN_PUBLIC_TUNNEL.")
    except BaseException:
        try:
            if workspace_tunnel:
                stop_process(workspace_tunnel)
        finally:
            if workspace_process:
                stop_process(workspace_process)
        raise

## Dừng khi dùng xong

In [ ]:
# Chỉ đổi True rồi chạy riêng cell này lúc đã dùng xong.
STOP_WORKSPACE = False
if STOP_WORKSPACE:
    try:
        if globals().get("workspace_tunnel"):
            stop_process(workspace_tunnel)
    finally:
        if globals().get("workspace_process"):
            stop_process(workspace_process)
    print("Đã dừng workspace và tunnel.")